# Train Model PM10 (XGBoost) & O3 (Random Forest)

- PM10 → XGBoost (mirip PM2.5, autokorelasi tinggi)
- O3 → Random Forest (pola fotokimia, bukan autokorelasi menit)

Jalankan semua cell, lalu copy metrik ke MLForecastPrototype.tsx

**Catatan:** Env vars `SUPABASE_URL` dan `SUPABASE_KEY` harus sudah di-set (atau ada di `.env`).

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from supabase import create_client, Client
import os
from datetime import datetime, timedelta
import joblib

SUPABASE_URL = os.getenv("SUPABASE_URL", "")
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

if not SUPABASE_URL or not SUPABASE_KEY:
    env_path = ".env"
    if os.path.exists(env_path):
        for line in open(env_path):
            if "=" in line and not line.startswith("#"):
                k, _, v = line.partition("=")
                os.environ[k.strip()] = v.strip().strip('"')
    SUPABASE_URL = os.getenv("SUPABASE_URL", "")
    SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

SUPABASE_URL = SUPABASE_URL.replace("http://", "https://")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print(f"Connected to {SUPABASE_URL}")

### 1. Ambil Data

In [ ]:
TABLE = "tb_konsentrasi_gas"
days_back = 14
since = (datetime.now() - timedelta(days=days_back)).isoformat()

all_data = []
offset = 0
while True:
    resp = supabase.table(TABLE) \
        .select("pm25_ugm3,pm10_ugm3,co_ugm3,no2_ugm3,o3_ugm3,temperature,humidity,created_at") \
        .gte("created_at", since) \
        .order("created_at", desc=False) \
        .range(offset, offset + 999) \
        .execute()
    batch = resp.data
    if not batch:
        break
    all_data.extend(batch)
    offset += len(batch)

df = pd.DataFrame(all_data)
for col in ["pm25_ugm3","pm10_ugm3","co_ugm3","no2_ugm3","o3_ugm3","temperature","humidity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
df = df.dropna(subset=["pm25_ugm3","pm10_ugm3","o3_ugm3","temperature","humidity"])
df = df.set_index("created_at").sort_index()
df = df[~df.index.duplicated(keep="first")]
df.rename(columns={"pm25_ugm3": "pm25", "pm10_ugm3": "pm10", "o3_ugm3": "o3"}, inplace=True)

print(f"Total: {len(df)} baris")
print(f"PM25:  mean={df['pm25'].mean():.1f}, std={df['pm25'].std():.1f}")
print(f"PM10:  mean={df['pm10'].mean():.1f}, std={df['pm10'].std():.1f}")
print(f"O3:    mean={df['o3'].mean():.1f}, std={df['o3'].std():.1f} ppb")

### 2. Feature Engineering (identik minutes_xgb.ipynb)

In [ ]:
RAW_COLS = ["pm25", "pm10", "co_ugm3", "no2_ugm3", "temperature", "humidity"]

feat = df.copy()

for col in RAW_COLS:
    if col not in feat.columns:
        continue
    feat[f"{col}_lag_1min"] = feat[col].shift(1)
    feat[f"{col}_lag_5min"] = feat[col].shift(5)
    feat[f"{col}_lag_15min"] = feat[col].shift(15)
    feat[f"{col}_lag_60min"] = feat[col].shift(60)
    feat[f"{col}_rolling_mean_5min"] = feat[col].rolling(window=5).mean()
    feat[f"{col}_rolling_std_5min"] = feat[col].rolling(window=5).std()
    feat[f"{col}_rolling_mean_15min"] = feat[col].rolling(window=15).mean()

feat["minute"] = feat.index.minute
feat["hour"] = feat.index.hour
feat["dayofweek"] = feat.index.dayofweek

print(f"Fitur: {len(feat.columns)} kolom, {len(feat)} baris (setelah dropna)")

### 3. Train & Evaluasi

In [ ]:
TARGETS = {
    "pm25": {"model": XGBRegressor, "params": {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "random_state": 42, "objective": "reg:squarederror", "verbosity": 0}, "pkl": "xgb_pm25.pkl"},
    "pm10": {"model": XGBRegressor, "params": {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "random_state": 42, "objective": "reg:squarederror", "verbosity": 0}, "pkl": "xgb_pm10.pkl"},
    "o3":  {"model": RandomForestRegressor, "params": {"n_estimators": 200, "max_depth": 10, "min_samples_split": 5, "random_state": 42}, "pkl": "rf_o3.pkl"},
}

FEATURE_COLS = [c for c in feat.columns if c not in ["pm25", "pm10", "o3"]]
print(f"Feature cols: {len(FEATURE_COLS)}")

results = {}

for target_name, config in TARGETS.items():
    print(f"\n{'='*60}")
    print(f"Training {config['model'].__name__} untuk {target_name.upper()}")
    print(f"{'='*60}")

    y = feat[target_name]
    X = feat[FEATURE_COLS].copy()

    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = config["model"](**config["params"])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    joblib.dump(model, config["pkl"])

    results[target_name] = {
        "model": config["model"].__name__,
        "pkl": config["pkl"],
        "mae": round(mae, 4),
        "rmse": round(rmse, 4),
        "r2": round(r2, 4),
        "train": len(X_train),
        "test": len(X_test),
    }

    print(f"  MAE  : {mae:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R2   : {r2:.4f} ({r2*100:.2f}%)")
    print(f"  PKL  : {config['pkl']}")

print(f"\n{'='*60}")
print("RINGKASAN SEMUA MODEL")
print(f"{'='*60}")
for name, r in results.items():
    print(f"  {name.upper():6s} | {r['model']:20s} | R2={r['r2']:.4f} ({r['r2']*100:.1f}%) | MAE={r['mae']:.4f} | {r['pkl']}")

### 4. Quick Test Prediksi

In [ ]:
feat_test = feat.dropna().iloc[[-1]]
X_row = feat_test[FEATURE_COLS]

print(f"\nNilai aktual saat ini:")
print(f"  PM2.5 : {feat_test['pm25'].values[0]:.1f} ug/m3")
print(f"  PM10  : {feat_test['pm10'].values[0]:.1f} ug/m3")
print(f"  O3    : {feat_test['o3'].values[0]:.1f} ppb")

print(f"\nPrediksi 1 menit ke depan:")
for target_name, config in TARGETS.items():
    model = joblib.load(config["pkl"])
    pred = float(model.predict(X_row.values)[0])
    print(f"  {target_name.upper():6s} : {pred:.1f} (actual: {feat_test[target_name].values[0]:.1f})")